## Just some basic EDA and a how-to guide. ##

In [49]:
import re
import os

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# "magic" command to make plots show up in the notebook
%matplotlib inline 

filepath = '../input/'

Kaggle changed they way they handle compressed files sometime in 2018.  They automatically uncompress them, and put them in a folder with the same name as the file (but all lowercased).  Here is where they are:

In [50]:
folders = os.listdir(filepath)
# Skip .xslx file, if we are ever able to upload it...
# Currently can't upload due to conflicts with other versions of the dataset on Kaggle.
folders = [f for f in folders if 'xlsx' not in f]
folders

['accepted_2007_to_2018q3.csv', 'rejected_2007_to_2018q3.csv']

In [51]:
os.listdir(filepath + folders[0])

['accepted_2007_to_2018Q3.csv']

In [52]:
acc_folder = filepath + [f for f in folders if 'accepted' in f][0]
accepted_fn = acc_folder + '/' + os.listdir(acc_folder)[0]

rej_folder = filepath + [f for f in folders if 'rejected' in f][0]
rejected_fn = rej_folder + '/' + os.listdir(rej_folder)[0]

accepted_fn

'../input/accepted_2007_to_2018q3.csv/accepted_2007_to_2018Q3.csv'

In [53]:
if os.path.isfile(accepted_fn) and os.path.isfile(rejected_fn):
    print('both paths still point to the actual file; all is good')
else:
    print('Kaggle changed how they handle compressed files again...you need to locate the files')

both paths still point to the actual file; all is good


In [54]:
# Takes a while to read, because these files are large...give it a minute or so
acc_df = pd.read_csv(accepted_fn)

# this is a dataset with rejected loans from lendingclub
rej_df = pd.read_csv(rejected_fn)

/opt/conda/lib/python3.6/site-packages/IPython/core/interactiveshell.py:2785: DtypeWarning: Columns (0,19,49,59,118,129,130,131,134,135,136,139,145,146,147) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [55]:
# this is how we can see how many entries are in the dataframe (df)
# it also works for numpy arrays
acc_df.shape

(2132287, 151)

In [56]:
rej_df.shape

(25054319, 9)

In [70]:
[col for col in acc_df.columns if 'fico' in col.lower()]

['fico_range_low',
 'fico_range_high',
 'last_fico_range_high',
 'last_fico_range_low',
 'sec_app_fico_range_low',
 'sec_app_fico_range_high']

In [71]:
# No fico score in rejected loans
[col for col in rej_df.columns if 'fico' in col.lower()]

[]

In [72]:
rej_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25054319 entries, 0 to 25054318
Data columns (total 9 columns):
Amount Requested        float64
Application Date        object
Loan Title              object
Risk_Score              float64
Debt-To-Income Ratio    object
Zip Code                object
State                   object
Employment Length       object
Policy Code             float64
dtypes: float64(3), object(6)
memory usage: 1.7+ GB


In [73]:
acc_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2132287 entries, 0 to 2132286
Columns: 151 entries, id to settlement_term
dtypes: float64(113), object(38)
memory usage: 2.4+ GB


In [57]:
# this is how many rows pandas will show by default with methods like pd.dataframe.head()
pd.options.display.max_rows

1000

In [58]:
# we want to increase it, because in this case there are a lot of column names
pd.options.display.max_rows = 1000

In [59]:
# the .T is transposing the matrix.
# We do this so the 111 column dataframe is easier to read (easier to scroll down than sideways) 
# .head() shows the first few rows of the data
acc_df.head().T

,0,1,2,3,4
id,93999587,96219354,96309331,96429334,96229241
member_id,NaN,NaN,NaN,NaN,NaN
loan_amnt,19200,7000,12000,6000,5000
funded_amnt,19200,7000,12000,6000,5000
funded_amnt_inv,19200,7000,12000,6000,5000
term,36 months,36 months,36 months,36 months,36 months
int_rate,13.99,5.32,16.99,7.49,10.49
installment,656.12,210.81,427.78,186.61,162.49
grade,C,A,D,A,B
sub_grade,C3,A1,D1,A4,B2


In [60]:
# .tail() shows the last few rows
acc_df.tail().T

,2132282,2132283,2132284,2132285,2132286
id,125893755,125340736,125551208,Total amount funded in policy code 1: 1741781700,Total amount funded in policy code 2: 564202131
member_id,NaN,NaN,NaN,NaN,NaN
loan_amnt,6000,12000,14000,NaN,NaN
funded_amnt,6000,12000,14000,NaN,NaN
funded_amnt_inv,6000,11975,13975,NaN,NaN
term,36 months,36 months,36 months,NaN,NaN
int_rate,10.91,10.42,13.59,NaN,NaN
installment,196.18,389.58,475.71,NaN,NaN
grade,B,B,C,NaN,NaN
sub_grade,B4,B3,C2,NaN,NaN


In [61]:
# .info() tells us the datatype(int64, `object` is a string)
# and will also tell us the number of non-null (not missing) data points for each column
# because this dataframe is so large, we have to force it to show the datatypes and non-null numbers with the arguments
acc_df.info(verbose=True, null_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2132287 entries, 0 to 2132286
Data columns (total 151 columns):
id                                            2132287 non-null object
member_id                                     0 non-null float64
loan_amnt                                     2132256 non-null float64
funded_amnt                                   2132256 non-null float64
funded_amnt_inv                               2132256 non-null float64
term                                          2132256 non-null object
int_rate                                      2132256 non-null float64
installment                                   2132256 non-null float64
grade                                         2132256 non-null object
sub_grade                                     2132256 non-null object
emp_title                                     1986234 non-null object
emp_length                                    1997053 non-null object
home_ownership                                

# Some common pandas functions

We find ourselves doing the same types of things often in pandas.  For example, masking.  

## Masking dataframes

Maybe we only want to see one column.  We do this like

`acc_df['int_rate']`

We can also subset a dataframe based on some critereon.  Let's look at only the highest interest rates

`acc_df[acc_df['int_rate'] > 20]`

We can take the mean, median, get the standard deviation of columns:

`acc_df['int_rate'].mean()`

In [ ]:
# shows some common summary statistics
# again, transposing with .T to make it easier to read
acc_df.describe().T

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
# "magic" command to make plots show up in the notebook
%matplotlib inline 

In [ ]:
acc_df[acc_df['int_rate'] > 20]['int_rate'].mean()
# thats a bit of a complicated statement.  Breaking it down:
# acc_df['int_rate'] > 20 returns a mask: an array of True/False values
# putting this array into acc_df[] returns the dataframe rows where the interest rates are greater than 20

# so this first part: acc_df[acc_df['int_rate'] > 20]
# gives us a dataframe

# we select a column with ['int_rate'] at the end.  Then we get the average value with .mean()

## Finding unique values

Numpy has a function for finding the unique values in an array:

np.unique(array)

This is built into pandas:

acc_df['grade'].unique()

shows us the unique values in that column.

In [ ]:
acc_df['grade'].unique()

In [ ]:
# selecting only grade A loans:
acc_df[acc_df['grade'] == 'A'].describe().T

In [ ]:
acc_df['loan_status'].unique()

In [ ]:
# looking at only defaulted loans:
default_categories = ['Default', 'Charged Off', 'Does not meet the credit policy. Status:Charged Off']
# .isin() is a trick for checking if something is in a list
# it's a pandas-specific function
acc_df[acc_df['loan_status'].isin(default_categories)].describe().T
# check out the average interest rate and dti (debt-to-income)

In [ ]:
# Seaborn is a recently-created Python library for easily making nice-looking plots
# You will have to install it with `conda install seaborn` or `pip install seaborn`, etc

# The docs are here for this function: http://seaborn.pydata.org/generated/seaborn.distplot.html
# Found by Googling 'seaborn histogram'
# Need to drop missing values, otherwise throws an error
# It would be better to impute missing values.
f = sns.distplot(acc_df['dti'].dropna())

In [ ]:
# outliers are screwing up the histogram... remove them
# adapted from http://stackoverflow.com/questions/11686720/is-there-a-numpy-builtin-to-reject-outliers-from-a-list
# we're using interquartile range to determine outliers
def reject_outliers(sr, iq_range=0.5, side='left', return_mask=False):
    """
    Takes an array (or pandas series) and returns an array with outliers excluded, according to the
    interquartile range.
    
    Parameters:
    -----------
    sr: array
        array of numeric values
    iq_range: float
        percent to calculate quartiles by, 0.5 will yield 25% and 75%ile quartiles
    side: string
        if 'left', will return everything below the highest quartile
        if 'right', will return everything above the lowest quartile
        if 'both', will return everything between the high and low quartiles
    """
    pcnt = (1 - iq_range) / 2
    qlow, median, qhigh = sr.dropna().quantile([pcnt, 0.50, 1-pcnt])
    iqr = qhigh - qlow
    if side=='both':
        mask = (sr - median).abs() <= iqr
    elif side=='left':
        mask = (sr - median) <= iqr
    elif side=='right':
        mask = (sr - median) >= iqr
    else:
        print('options for side are left, right, or both')
    
    if return_mask:
        return mask
    
    return sr[mask]

In [ ]:
# sweeeeeeeetttt....
dti_no_outliers = reject_outliers(acc_df['dti'], iq_range=0.85) # arrived at 0.85 via trial and error
f = sns.distplot(dti_no_outliers)
# other types of plot examples:
# http://seaborn.pydata.org/examples/

In [ ]:
# sets the xkcd style if you want to make the plots look funny...may neet to install some fonts
plt.xkcd()

In [ ]:
f = sns.distplot(dti_no_outliers)

In [ ]:
# back to default
import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)

In [ ]:
f = sns.distplot(dti_no_outliers)

In [ ]:
# other styles:
# http://matplotlib.org/users/style_sheets.html
# this is the R ggplot style, which some people really like
plt.style.use('ggplot')

In [ ]:
# http://seaborn.pydata.org/generated/seaborn.regplot.html
# takes a long time because there are a lot of points, but works for smaller datasets
# f = sns.regplot(data=acc_df, x='dti', y='int_rate', fit_reg=False)
# instead, lets make a heatmap:
# http://stackoverflow.com/questions/2369492/generate-a-heatmap-in-matplotlib-using-a-scatter-data-set
# http://seaborn.pydata.org/generated/seaborn.jointplot.html
mask = reject_outliers(acc_df['dti'], iq_range=0.85, return_mask=True)
f = sns.jointplot(data=acc_df.ix[mask, :], x='dti', y='int_rate', kind='hex', joint_kws=dict(gridsize=50))